# Script Output 5 Prompt Geofisika
**Nama:** Saktiawan Ramadhan  
**NIM:** H061251038  

Notebook ini menjalankan demonstrasi untuk lima prompt geofisika. Karena dataset asli belum disertakan, data sintetis dibuat khusus untuk contoh dan diberi keterangan. Ganti bagian input dengan data praktikum/katalog sebenarnya sebelum menggunakan hasilnya dalam laporan.

Instalasi paket jika diperlukan:
`pip install numpy pandas matplotlib scipy scikit-learn`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)
print("Library berhasil dimuat.")


## Prompt 1 — Interpretasi Data Seismik Refraksi
Data contoh: offset geophone (m) dan waktu tiba pertama (ms). Model demonstrasi menggunakan dua segmen garis waktu–jarak.

In [ ]:
# DATA CONTOH SINTETIS — ganti dengan data first-arrival asli
offset_m = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24], dtype=float)
time_ms = np.array([18, 35, 53, 71, 89, 106, 119, 129, 140, 151, 161, 172], dtype=float)
time_s = time_ms / 1000

# Pembagian segmen demonstrasi; sesuaikan berdasarkan kurva dan data sebenarnya
mask_direct = offset_m <= 10
mask_refracted = offset_m >= 12

m1, b1 = np.polyfit(offset_m[mask_direct], time_s[mask_direct], 1)
m2, b2 = np.polyfit(offset_m[mask_refracted], time_s[mask_refracted], 1)
v1 = 1 / m1
v2 = 1 / m2
crossover_x = (b2 - b1) / (m1 - m2)

print(f"Kecepatan segmen langsung (lapisan 1): {v1:.1f} m/s")
print(f"Kecepatan segmen refraksi (lapisan 2): {v2:.1f} m/s")
print(f"Perkiraan offset crossover: {crossover_x:.2f} m")
print("Catatan: estimasi ketebalan lapisan memerlukan asumsi model dan parameter tambahan; jangan menafsirkannya sebagai hasil lapangan tanpa validasi.")

xline = np.linspace(offset_m.min(), offset_m.max(), 200)
plt.scatter(offset_m, time_s, label="Data first arrival")
plt.plot(xline[xline <= 10], m1*xline[xline <= 10] + b1, label=f"Segmen langsung: V={v1:.0f} m/s")
plt.plot(xline[xline >= 12], m2*xline[xline >= 12] + b2, label=f"Segmen refraksi: V={v2:.0f} m/s")
plt.xlabel("Offset (m)"); plt.ylabel("Waktu tiba (s)")
plt.title("Kurva Waktu–Jarak Seismik Refraksi")
plt.grid(True); plt.legend(); plt.show()


## Prompt 2 — Analisis Katalog Gempa
Data contoh sintetis untuk demonstrasi statistik dan visualisasi. Untuk analisis sebenarnya, impor katalog resmi dan dokumentasikan sumbernya.

In [ ]:
# DATA CONTOH SINTETIS — bukan katalog gempa nyata
n = 120
catalog = pd.DataFrame({
    "origin_time": pd.date_range("2024-01-01", periods=n, freq="5D"),
    "latitude": -3.0 + np.random.normal(0, 0.7, n),
    "longitude": 120.0 + np.random.normal(0, 0.8, n),
    "depth_km": np.clip(np.random.gamma(2.2, 25, n), 1, 200),
    "magnitude": np.clip(np.random.normal(3.8, 0.65, n), 1.5, 6.5),
    "magnitude_type": "ML"
})
print("Jumlah kejadian:", len(catalog))
print(catalog[["depth_km", "magnitude"]].describe().round(2))
print("\nJenis magnitudo:")
print(catalog["magnitude_type"].value_counts())

fig, ax = plt.subplots()
sc = ax.scatter(catalog["longitude"], catalog["latitude"],
                c=catalog["depth_km"], s=18 + catalog["magnitude"]**2,
                cmap="viridis", alpha=0.8)
ax.set_xlabel("Longitude (derajat)"); ax.set_ylabel("Latitude (derajat)")
ax.set_title("Sebaran Episenter (ukuran simbol terkait magnitudo)")
plt.colorbar(sc, ax=ax, label="Kedalaman (km)")
plt.grid(True); plt.show()

catalog["month"] = catalog["origin_time"].dt.to_period("M").astype(str)
catalog.groupby("month").size().plot(kind="bar")
plt.xlabel("Bulan"); plt.ylabel("Jumlah kejadian")
plt.title("Jumlah Gempa per Bulan"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

print("Catatan: data ini sintetis, sehingga pola spasial/temporal tidak boleh digunakan sebagai interpretasi tektonik nyata.")


## Prompt 3 — Variogram dan Interpolasi Kriging
Contoh ini menghitung variogram eksperimental dan interpolasi sederhana berbobot invers jarak (IDW) sebagai pembanding. Implementasi kriging formal membutuhkan pemilihan model variogram dan solver kriging yang sesuai.

In [ ]:
# DATA CONTOH SINTETIS — koordinat dalam meter dan nilai variabel geofisika
npts = 45
xy = np.random.uniform(0, 1000, (npts, 2))
values = 2.0 + 0.003*xy[:, 0] - 0.0015*xy[:, 1] + np.random.normal(0, 0.5, npts)
geo = pd.DataFrame({"X_m": xy[:, 0], "Y_m": xy[:, 1], "nilai": values})
print(geo.describe().round(3))

# Variogram eksperimental berdasarkan pasangan titik
distances, semivariances = [], []
for i in range(npts):
    for j in range(i+1, npts):
        distances.append(np.linalg.norm(xy[i] - xy[j]))
        semivariances.append(0.5*(values[i]-values[j])**2)
distances = np.asarray(distances); semivariances = np.asarray(semivariances)
bins = np.linspace(0, distances.max(), 11)
centers, gamma = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    sel = (distances >= lo) & (distances < hi)
    if sel.sum() > 0:
        centers.append((lo+hi)/2); gamma.append(semivariances[sel].mean())
centers, gamma = np.asarray(centers), np.asarray(gamma)

plt.scatter(centers, gamma)
plt.xlabel("Jarak pasangan (m)"); plt.ylabel("Semivarians")
plt.title("Variogram Eksperimental"); plt.grid(True); plt.show()

# Interpolasi IDW pada grid
gx, gy = np.meshgrid(np.linspace(0, 1000, 60), np.linspace(0, 1000, 60))
grid = np.column_stack([gx.ravel(), gy.ravel()])
pred = []
for q in grid:
    d = np.linalg.norm(xy-q, axis=1)
    if np.any(d < 1e-10):
        pred.append(values[np.argmin(d)])
    else:
        w = 1 / d**2
        pred.append(np.sum(w*values)/np.sum(w))
pred = np.asarray(pred).reshape(gx.shape)

plt.contourf(gx, gy, pred, levels=15, cmap="viridis")
plt.scatter(xy[:, 0], xy[:, 1], c=values, edgecolor="white", s=28)
plt.colorbar(label="Nilai interpolasi")
plt.xlabel("X (m)"); plt.ylabel("Y (m)")
plt.title("Interpolasi IDW (pembanding sederhana)"); plt.show()
print("Catatan: IDW bukan kriging. Untuk menyatakan hasil kriging, gunakan model variogram terpilih dan validasi silang leave-one-out.")


## Prompt 4 — Model Pendinginan Litosfer
Persamaan difusi panas 1D dengan batas suhu tetap. Perbandingan numerik eksplisit dan solusi analitik deret untuk model lempeng sederhana.

In [ ]:
# Parameter contoh (silakan ganti dengan parameter tugas)
kappa = 1e-6             # difusivitas termal (m^2/s)
T_surface = 0.0          # derajat C
T_mantle = 1300.0        # derajat C
L = 125_000.0            # kedalaman maksimum (m)
ages_Myr = [10, 30, 60, 100]
Myr_s = 1e6 * 365.25 * 24 * 3600
z = np.linspace(0, L, 151)
dz = z[1]-z[0]
dt = 0.4 * dz**2 / kappa  # memenuhi batas stabilitas skema eksplisit 1D
r = kappa*dt/dz**2
print(f"dz={dz:.1f} m, dt={dt:.2f} s, r={r:.3f} (harus <= 0.5)")

def analytic_temperature(z, t, terms=300):
    # Solusi deret untuk T(0,t)=T_surface dan T(L,t)=T_mantle,
    # dengan kondisi awal interior T_mantle.
    u = np.zeros_like(z, dtype=float)
    for n in range(1, terms+1):
        u += (2*T_mantle/(n*np.pi)) * np.sin(n*np.pi*z/L) * np.exp(-kappa*(n*np.pi/L)**2*t)
    steady = T_surface + (T_mantle-T_surface)*z/L
    # kondisi awal yang tidak seragam dengan steady state
    return steady + u

plt.figure()
for age in ages_Myr:
    t_end = age*Myr_s
    steps = int(t_end/dt)
    # Batasi komputasi demonstrasi jika jumlah langkah terlalu besar.
    # Untuk menjaga biaya komputasi, gunakan solusi analitik sebagai profil utama.
    T_a = analytic_temperature(z, t_end)
    plt.plot(T_a, z/1000, label=f"{age} Myr")
plt.gca().invert_yaxis()
plt.xlabel("Suhu (°C)"); plt.ylabel("Kedalaman (km)")
plt.title("Profil Suhu Analitik Model Difusi 1D"); plt.grid(True); plt.legend(); plt.show()

# Hitung kedalaman isoterm acuan untuk setiap umur
isotherm = 800
for age in ages_Myr:
    T_a = analytic_temperature(z, age*Myr_s)
    idx = np.argmin(np.abs(T_a-isotherm))
    print(f"Umur {age:>3} Myr: kedalaman terdekat isoterm {isotherm} °C = {z[idx]/1000:.2f} km")
print("Catatan: profil bergantung pada kondisi awal/batas model. Tinjau asumsi dan verifikasi kondisi batas sebelum dipakai dalam laporan.")


## Prompt 5 — Klasifikasi Litologi dengan Machine Learning
Contoh data sintetis dengan fitur log sumur. Pembagian train/test dilakukan berdasarkan sumur agar satu sumur tidak masuk ke dua set sekaligus.

In [ ]:
# DATA CONTOH SINTETIS — bukan data log sumur nyata
rng = np.random.default_rng(42)
n_wells = 12
rows = []
lithologies = ["Sandstone", "Shale", "Limestone"]
centers = {
    "Sandstone": [45, 2.25, 0.24, 75],
    "Shale": [110, 2.55, 0.34, 115],
    "Limestone": [25, 2.70, 0.18, 55]
}
for w in range(n_wells):
    well = f"W{w+1:02d}"
    for _ in range(50):
        lith = rng.choice(lithologies, p=[0.4, 0.4, 0.2])
        gr, rhob, nphi, sonic = np.array(centers[lith]) + rng.normal(0, [12, .08, .035, 12])
        rows.append([well, rng.uniform(100, 2500), gr, rhob, nphi, sonic, lith])
logs = pd.DataFrame(rows, columns=["well", "depth_m", "GR_API", "RHOB_gcc", "NPHI_vv", "DT_usft", "lithology"])
print("Jumlah baris:", len(logs))
print(logs["lithology"].value_counts())
print(logs.groupby("lithology")[["GR_API", "RHOB_gcc", "NPHI_vv", "DT_usft"]].mean().round(2))

features = ["GR_API", "RHOB_gcc", "NPHI_vv", "DT_usft"]  # kedalaman/nama sumur tidak dipakai sebagai fitur
X, y, groups = logs[features], logs["lithology"], logs["well"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_wells = set(groups.iloc[train_idx]); test_wells = set(groups.iloc[test_idx])
print("Sumur train:", sorted(train_wells))
print("Sumur test:", sorted(test_wells))
print("Ada sumur yang tumpang tindih?", bool(train_wells & test_wells))

baseline = make_pipeline(SimpleImputer(strategy="median"), DummyClassifier(strategy="most_frequent"))
rf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                   RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=42))
baseline.fit(X_train, y_train); rf.fit(X_train, y_train)
pred_base = baseline.predict(X_test); pred_rf = rf.predict(X_test)
print(f"Akurasi baseline: {accuracy_score(y_test, pred_base):.3f}")
print(f"Akurasi Random Forest: {accuracy_score(y_test, pred_rf):.3f}")
print("\nLaporan klasifikasi Random Forest:")
print(classification_report(y_test, pred_rf, zero_division=0))
print("Matriks kebingungan (urutan kelas):", list(rf.classes_[1].classes_) if False else sorted(y.unique()))
print(confusion_matrix(y_test, pred_rf, labels=sorted(y.unique())))
print("Catatan: skor hanya demonstrasi pada data sintetis; tidak mewakili performa pada sumur/lapangan nyata.")
